# AI Coding Foundation Lab

**課程**: Session 1 - Technical Foundations  
**目標**: 理解 Token 與 Embedding，建立 AI 技術基礎

---

## 📚 Lab 結構

- **Part 1**: 環境設定
- **Part 2**: Tokenization 實驗
- **Part 3**: Embedding 與向量相似度

---

## Part 1: 環境設定

### 🎯 目標
安裝必要的套件，確保實驗環境正常運行

In [ ]:
# 安裝必要的套件
!pip install tiktoken sentence-transformers scikit-learn numpy -q

print("✓ 套件安裝完成！")

In [ ]:
# 匯入必要的函式庫
import tiktoken
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("✓ 函式庫匯入成功！")

---

## Part 2: Tokenization 實驗

### 🎯 目標

- 理解 Token 是 AI 處理文字的最小單位
- 觀察中英文的 Token 差異
- 計算專案的 Token 成本

---

### Task 2.1: 基本 Tokenization

**概念**: Token 是 AI 處理文字的最小單位，不是字元，也不完全是單字。

In [ ]:
# 初始化 GPT-4 使用的 tokenizer
enc = tiktoken.get_encoding("cl100k_base")

# 實驗 1: 英文
text_en = "Hello World"
tokens_en = enc.encode(text_en)
print(f"英文範例: '{text_en}'")
print(f"Token 數量: {len(tokens_en)}")
print(f"Token IDs: {tokens_en}")
print()

# 實驗 2: 中文
text_zh = "你好世界"
tokens_zh = enc.encode(text_zh)
print(f"中文範例: '{text_zh}'")
print(f"Token 數量: {len(tokens_zh)}")
print(f"Token IDs: {tokens_zh}")
print()

# 觀察
print("📊 觀察結果:")
print(f"   - 英文 '{text_en}' = {len(tokens_en)} tokens")
print(f"   - 中文 '{text_zh}' = {len(tokens_zh)} tokens")
print(f"   - 中文成本是英文的 {len(tokens_zh)/len(tokens_en):.1f} 倍！")

### Task 2.2: 計算專案成本

**實務應用**: 了解不同規模專案的 Token 成本

In [ ]:
# LLM pricing (per million tokens)
# https://www.llm-prices.com/
# https://openai.com/api/pricing/
# https://www.anthropic.com/pricing#anthropic-api

MODEL_PRICING = {
  'gpt-4o': {'input': 2.5/1_000_000, 'output': 10/1_000_000},
  'gpt-4o-mini': {'input': 0.15/1_000_000, 'output': 0.6/1_000_000},
  'claude-sonnet-4.5': {'input': 3/1_000_000, 'output': 15/1_000_000},
  'claude-haiku-4.5': {'input': 1/1_000_000, 'output': 5/1_000_000},
  'gemini-2.5-pro': {'input': 1.25/1_000_000, 'output': 10/1_000_000},
  'gemini-2.5-flash': {'input': 0.15/1_000_000, 'output': 0.6/1_000_000},
}

# Helper function
def calculate_cost(input_tokens, model_name, output_ratio=0.15):
  """
  計算完整的 LLM 使用成本（包含 input + output）

  Args:
      input_tokens: 輸入的 token 數量
      model_name: 模型名稱 (如 'gpt-4o', 'claude-sonnet-4.5')
      output_ratio: output tokens 占 input tokens 的比例 (預設 0.15)

  Returns:
      總成本 (美元)
  """
  if model_name not in MODEL_PRICING:
      raise ValueError(f"未知的模型: {model_name}. 可用模型: {list(MODEL_PRICING.keys())}")

  prices = MODEL_PRICING[model_name]
  output_tokens = input_tokens * output_ratio

  input_cost = input_tokens * prices['input']
  output_cost = output_tokens * prices['output']

  return input_cost + output_cost

print("✓ Created Helper function！")

In [ ]:
# 模擬不同規模的專案程式碼

# Demo 專案範例 (3 個檔案)
small_project = """
// app.js
const express = require('express');
const app = express();
app.listen(3000);

// routes.js
module.exports = {
  getUser: (req, res) => res.json({ user: 'John' })
};

// config.js
module.exports = { port: 3000, db: 'mongodb://localhost' };
"""

# 計算 tokens
small_tokens = len(enc.encode(small_project))

# 假設 output tokens 是 input 的 15%
output_ratio = 0.15

print("📊 專案 Token 成本分析（含 Input + Output）")
print("=" * 60)
print(f"假設: Output tokens = {output_ratio*100:.0f}% of Input tokens\n")

# 定義專案規模
projects = [
    ("Demo project (3 個檔案)", small_tokens),
    ("估算 60 個檔案", small_tokens * 20),
    ("估算 300 個檔案", small_tokens * 100),
]

# 遍歷每個專案規模
for project_name, tokens in projects:
    print(f"{project_name}:")
    print(f"  - Input Tokens: {tokens:,}")
    print(f"  - Output Tokens (估算): {int(tokens * output_ratio):,}")

    # 遍歷所有模型
    for model_name in MODEL_PRICING.keys():
        cost = calculate_cost(tokens, model_name, output_ratio)
        # 根據成本大小決定顯示格式
        if cost < 0.0001:
            print(f"  - {model_name}: ${cost:.6f}")
        elif cost < 0.1:
            print(f"  - {model_name}: ${cost:.4f}")
        else:
            print(f"  - {model_name}: ${cost:.2f}")

    print()

print("💡 關鍵洞察:")
print("   1. Output tokens 雖然少，但價格高 (5-10倍)")
print("   2. 大專案無法全部塞進 Context Window (128K-200K tokens)")
print("   3. ➜ 這就是為什麼需要 RAG！")

# 計算大專案的成本比較
large_tokens = small_tokens * 100
print("\n📌 大專案成本比較:")
costs = {model: calculate_cost(large_tokens, model) for model in MODEL_PRICING.keys()}
min_cost = min(costs.values())

for model, cost in costs.items():
    ratio = cost / min_cost
    print(f"   {model}: ${cost:.4f} ({ratio:.1f}x)")

### Task 2.3: 實驗區

**動手試試**: 用自己的文字實驗

In [ ]:
your_text = "在這裡輸入你想測試的文字"
model_name = "claude-sonnet-4.5"

tokens = enc.encode(your_text)
token_count = len(tokens)
cost = calculate_cost(token_count, model_name)
print(f"你的文字: '{your_text}'")
print(f"Token 數量: {len(tokens)}")
print(f"Cost ({model_name}): ${cost:.6f}")

In [ ]:
# 比較英文註解 vs 中文註解
code_with_en_comments = """
// This function calculates the sum of two numbers
function add(a, b) {
  return a + b;
}
"""

code_with_zh_comments = """
// 這個函式計算兩個數字的總和
function add(a, b) {
  return a + b;
}
"""

en_tokens = len(enc.encode(code_with_en_comments))
zh_tokens = len(enc.encode(code_with_zh_comments))

print("📊 英文註解 vs 中文註解")
print(f"英文註解: {en_tokens} tokens")
print(f"中文註解: {zh_tokens} tokens")
print(f"差異: {zh_tokens - en_tokens} tokens (+{(zh_tokens/en_tokens - 1)*100:.1f}%)")

### ✅ Part 2 完成檢查點

**你學到了**:
- ✓ Token 是 AI 處理文字的最小單位
- ✓ 中文比英文消耗更多 tokens (約 1.5-2 倍)
- ✓ 大專案的 token 成本可能很高
- ✓ 為什麼需要 RAG 來管理大型 codebase

**接下來**: Part 3 - Embedding 與向量相似度

---

## Part 3: Embedding 與向量相似度

### 🎯 目標

- 理解 Embedding 如何將文字轉換為向量
- 學習如何計算語義相似度
- 理解為什麼向量搜尋比關鍵字搜尋更強大

---

### Task 3.1: 文字轉向量

---



**概念**: Embedding 將文字轉換為數字向量，編碼語義資訊

In [ ]:
# 載入 Sentence Transformer 模型
print("載入模型中...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✓ 模型載入完成！\n")

In [ ]:
# 將文字轉換為向量
text = "I love programming"
embedding = model.encode(text)

print(f"原始文字: '{text}'")
print(f"向量維度: {len(embedding)}")
print(f"\n前 10 個數字:")
print(embedding[:10])
print("\n💡 這 384 個數字就是 AI「理解」這句話的方式！")

### Task 3.2: 計算語義相似度

**核心概念**: 語義相近的文字，向量也相近

In [ ]:
# 準備測試句子
sentences = [
    "I love programming",
    "I enjoy coding",
    "The weather is nice today",
    "我喜歡寫程式"
]

# 轉換為向量
embeddings = model.encode(sentences)

# 計算相似度矩陣
similarity_matrix = cosine_similarity(embeddings)

# 顯示結果
print("📊 語義相似度分析\n")
print("=" * 70)

for i, sent1 in enumerate(sentences):
    for j, sent2 in enumerate(sentences):
        if i < j:  # 只顯示上三角矩陣
            score = similarity_matrix[i][j]
            print(f"\n句子 {i+1}: {sent1}")
            print(f"句子 {j+1}: {sent2}")
            print(f"相似度: {score:.3f} {'✓ 高度相關' if score > 0.7 else '⊗ 不相關' if score < 0.3 else '~ 中度相關'}")
            print("-" * 70)

#### 📊 結果分析

**關鍵觀察**:

1. **"I love programming" vs "I enjoy coding"**
   - 相似度應該 > 0.7
   - 用詞完全不同，但語義相近！

2. **"I love programming" vs "The weather is nice"**
   - 相似度應該 < 0.3
   - 完全無關的主題

3. **"I love programming" vs "我喜歡寫程式"**
   - 相似度應該 > 0.7
   - 等等! 觀察一下相似度❗


#### 🚫 問題出在哪？
all-MiniLM-L6-v2 這個模型主要是用英文訓練的！

**模型看到中文時：**

- ❌ 沒有好好學過中文
- ❌ 不知道「我喜歡寫程式」是什麼意思
- ❌ 把中文當成「陌生的符號」處理

結果：隨便給了一個低分數

**💡 類比說明**

就像你找一個只會英文的人：

- ✅ "I love programming" vs "I enjoy coding" → 他知道很像
- ❌ "I love programming" vs "我喜歡寫程式" → 他完全看不懂中文，只能說「不知道」

**✅ 解決方案**

需要換成支援多語言的模型！

In [ ]:
# 載入多語言模型
print("載入多語言模型中...")
multilingual_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("✓ 多語言模型載入完成！\n")

In [ ]:
# 準備測試句子
sentences = [
    "I love programming",
    "I enjoy coding",
    "The weather is nice today",
    "我喜歡寫程式"
]

# 轉換為向量
embeddings = multilingual_model.encode(sentences)

# 計算相似度矩陣
similarity_matrix = cosine_similarity(embeddings)

# 顯示結果
print("📊 語義相似度分析\n")
print("=" * 70)

for i, sent1 in enumerate(sentences):
    for j, sent2 in enumerate(sentences):
        if i < j:  # 只顯示上三角矩陣
            score = similarity_matrix[i][j]
            print(f"\n句子 {i+1}: {sent1}")
            print(f"句子 {j+1}: {sent2}")
            print(f"相似度: {score:.3f} {'✓ 高度相關' if score > 0.7 else '⊗ 不相關' if score < 0.3 else '~ 中度相關'}")
            print("-" * 70)

**預期結果**:
"I love programming" vs "我喜歡寫程式"
- 相似度 > 0.7 ✅



#### 🔍 為什麼會有這個差異？

#### 訓練資料的差異

**英文模型** (all-MiniLM-L6-v2):
```
訓練資料 = 99% 英文句子
學會了 → "programming" = "coding"
沒學過 → "programming" ≠ "程式"
```

**多語言模型** (multilingual):
```
訓練資料 = 英文 + 中文 + 法文 + ...
學會了 → "programming" = "coding" = "程式" = "プログラミング"
```

Reference
- https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
- https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

### Task 3.3: 實驗區

**動手試試**: 測試你自己的句子

In [ ]:
# 試試看找兩個語義相近但用詞完全不同的句子
your_sentences = [
    "狗狗",  # ← 修改這裡
    "犬",   # ← 修改這裡
]

# 計算相似度
# 選擇使用 model or multilingual_model
your_embeddings = multilingual_model.encode(your_sentences)
similarity = cosine_similarity([your_embeddings[0]], [your_embeddings[1]])[0][0]

print(f"句子 1: {your_sentences[0]}")
print(f"句子 2: {your_sentences[1]}")
print(f"\n相似度: {similarity:.3f}")

if similarity > 0.7:
    print("✓ 高度相關！語義非常相近")
elif similarity > 0.4:
    print("~ 中度相關，有一些共同點")
else:
    print("⊗ 語義差異很大")

In [ ]:
# 技術術語的語義理解
tech_terms = [
    "API endpoint",
    "REST interface",
    "GraphQL query",
    "資料庫查詢"
]

# 選擇使用 model or multilingual_model
tech_embeddings = multilingual_model.encode(tech_terms)
tech_similarity = cosine_similarity(tech_embeddings)

print("📊 技術術語的語義相似度\n")
for i, term1 in enumerate(tech_terms):
    for j, term2 in enumerate(tech_terms):
        if i < j:
            score = tech_similarity[i][j]
            print(f"{term1:20s} <-> {term2:20s} : {score:.3f}")

💡 **這就是 RAG 檢索的基礎：用向量相似度找到語義相關的文件！**

### ✅ Part 3 完成檢查點

**你學到了**:
- ✓ Embedding 將文字轉換為 384 維向量
- ✓ 語義相近的文字，向量也相近
- ✓ Cosine Similarity 可以測量語義相似度
- ✓ 向量搜尋可以理解同義詞和跨語言(有條件!!跨語言需要具備多語言能力的模型)
- ✓ **這就是 RAG 和 @codebase 的核心技術！**

---



## ➡️ 下一步

向量搜尋與 RAG

---